In [1]:
!pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 20.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter co

In [2]:
import os
import joblib
import warnings
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
from sklearn.svm import SVR
import matplotlib.pyplot as plt
from google.colab import files
from statsmodels.tsa.arima.model import ARIMA
from autogluon.tabular import TabularPredictor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [3]:
# Open a file upload dialog
uploaded = files.upload()

Saving Cleaned Dataset.csv to Cleaned Dataset.csv


In [4]:
# Import dataset
dataset = pd.read_csv("Cleaned Dataset.csv")
dataset

,Country,Year,BEV Percentage (Total Number Of Registrations),BEV (New Registrations),GDP,CPI,EG,Recharging Points,AC Recharging Speed (km/h),DC Recharging Speed (km/h),...,Real Range,Purchase price (EUR),Log_BEV Percentage (Total Number Of Registrations),Log_BEV (New Registrations),Log_Recharging Points,Log_GDP,Log_CPI,Log_Available,Log_DC Recharging Speed (km/h),YJ_Purchase price (EUR)
0,Austria,2011,0.18,631.0,308167.0,3.286579,1.6,963,17.500000,175.000000,...,95.000000,30469.93,0.165514,6.448889,6.871091,12.638400,1.455489,1.609438,5.170484,-1.493084
1,Austria,2012,0.13,427.0,316589.4,2.485676,1.0,1063,29.169521,217.394139,...,95.000000,30469.93,0.122218,6.059123,6.969791,12.665364,1.248662,1.609438,5.386301,-1.493084
2,Austria,2013,0.21,656.0,321191.7,2.000156,0.4,1173,48.200000,270.000000,...,142.220000,30469.93,0.190620,6.487684,7.068172,12.679797,1.098664,2.197225,5.602119,-1.493084
3,Austria,2014,0.44,1344.0,330113.5,1.605812,1.0,1393,22.600000,240.000000,...,155.770000,35555.64,0.364643,7.204149,7.239933,12.707195,0.957744,2.639057,5.484797,-0.885657
4,Austria,2015,0.55,1699.0,342083.5,0.896563,0.6,1625,47.700000,381.250000,...,215.220000,42052.00,0.438255,7.438384,7.393878,12.742813,0.640043,3.091042,5.946075,0.277809
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,Sweden,2020,9.55,27897.0,5012855.0,0.497367,-1.3,47680,39.325301,403.164557,...,286.200000,38311.38,2.356126,10.236310,10.772288,15.427516,0.403708,4.584967,6.001822,-0.451036
374,Sweden,2021,19.09,57454.0,5417760.0,2.163197,1.3,70537,40.904348,493.391304,...,308.730000,40745.01,3.000222,10.958757,11.163907,15.505193,1.151583,5.129899,6.203327,0.003737
375,Sweden,2022,32.84,94603.0,5816415.0,8.369291,3.5,83510,49.314815,594.259259,...,345.440000,43874.27,3.521644,11.457455,11.332734,15.576195,2.237437,5.537334,6.388997,0.697159
376,Sweden,2023,38.42,111338.0,6143187.0,8.548625,1.2,126758,70.307692,649.230769,...,361.794521,45999.05,3.674273,11.620335,11.750043,15.630854,2.256397,5.918894,6.477327,1.244323


In [5]:
# Define the target
target = 'Log_BEV Percentage (Total Number Of Registrations)'

In [7]:
# Load the PCA tools
tools = joblib.load("pca_processors.pkl")
scaler = tools['scaler']
pca = tools['pca_model']
features = tools['feature_names']

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
# Data splitting into train data and test data
pre_covid_df = dataset[dataset['Year'] < 2023].copy()
post_covid_df = dataset[dataset['Year'] >= 2023].copy()

Please refer the jupyter notebook - **3.1.Feature Selection_BEV Percentage (Primary Target)** to understand the imputation

In [9]:
# Check which columns have NaNs and how many
print("Missing Values Per Column on Train data (2011-2022)")
print(pre_covid_df[features].isna().sum())

print("\nMissing Values Per Column Test data (2023-2024)")
print(post_covid_df[features].isna().sum())

Missing Values Per Column on Train data (2011-2022)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           9
dtype: int64

Missing Values Per Column Test data (2023-2024)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           0
dtype: int64


In [11]:
# Checking which rows are empty
print("Missing Log_CPI in Train data (2011-2022)")
print(pre_covid_df[pre_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])

Missing Log_CPI in Train data (2011-2022)
    Country  Year  Log_CPI
0  Bulgaria  2014      NaN
1   Croatia  2016      NaN
2    Cyprus  2014      NaN
3    Cyprus  2015      NaN
4    Cyprus  2016      NaN
5    Greece  2014      NaN
6    Greece  2015      NaN
7    Greece  2020      NaN
8   Romania  2016      NaN


In [12]:
# Imputation for Log_CPI NaNs by filling the spaces with the median

# Calculate the median Log_CPI for every country of train data
country_medians = pre_covid_df.groupby('Country')['Log_CPI'].median()

# Function to fill the gaps
def impute_by_country(df):
    df_clean = df.copy()

    # Loop through the data, if Log_CPI is missing, we look up that specific country's median from our 'country_medians' list.
    df_clean['Log_CPI'] = df_clean['Log_CPI'].fillna(df_clean['Country'].map(country_medians))

    return df_clean

# Apply the fix to both datasets
pre_covid_fixed = impute_by_country(pre_covid_df)
post_covid_fixed = impute_by_country(post_covid_df)

# Extract 10 features
X_train_imputed = pre_covid_fixed[features]
X_test_imputed = post_covid_fixed[features]

In [13]:
# Verfication
print("Missing Values Per Column on Train data (2011-2022)")
print(pre_covid_fixed[features].isna().sum())

print("\nMissing Values Per Column Test data (2023-2024)")
print(post_covid_fixed[features].isna().sum())

Missing Values Per Column on Train data (2011-2022)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           0
dtype: int64

Missing Values Per Column Test data (2023-2024)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           0
dtype: int64


In [14]:
# Scale the features using the scaler loaded from the .pkl file
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [15]:
# Transform the features into 3 Principal Components using the loaded PCA model
# and convert them back to a dataframe for easy handling
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=['PC1', 'PC2', 'PC3'], index=pre_covid_fixed.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_scaled), columns=['PC1', 'PC2', 'PC3'], index=post_covid_fixed.index)

In [16]:
# Define the targets
y_train = pre_covid_fixed[target]
y_test = post_covid_fixed[target]

print(f"Train Shape (PCs): {X_train_pca.shape}")
print(f"Test Shape (PCs): {X_test_pca.shape}")

Train Shape (PCs): (324, 3)
Test Shape (PCs): (54, 3)


In [17]:
# Define the clusters based on the K-Means clusters + PCA
leaders_list = [
    'Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland', 'France', 'Germany', 'Hungary',
    'Italy', 'Netherlands', 'Poland', 'Romania', 'Spain', 'Sweden'
]

followers_list = [
    'Bulgaria', 'Croatia', 'Cyprus', 'Estonia', 'Greece', 'Ireland', 'Latvia', 'Lithuania',
    'Luxembourg', 'Malta', 'Portugal', 'Slovakia', 'Slovenia'
]

In [18]:
# Filter Training Data 
X_train_leaders = X_train_pca[pre_covid_fixed['Country'].isin(leaders_list)]
y_train_leaders = y_train[pre_covid_fixed['Country'].isin(leaders_list)]

X_train_followers = X_train_pca[pre_covid_fixed['Country'].isin(followers_list)]
y_train_followers = y_train[pre_covid_fixed['Country'].isin(followers_list)]

# Filter Testing Data 
X_test_leaders = X_test_pca[post_covid_fixed['Country'].isin(leaders_list)]
y_test_leaders = y_test[post_covid_fixed['Country'].isin(leaders_list)]

X_test_followers = X_test_pca[post_covid_fixed['Country'].isin(followers_list)]
y_test_followers = y_test[post_covid_fixed['Country'].isin(followers_list)]

## AUTOGLUON MODEL

**1. Explanation:** [AutoGluon](https://aws.amazon.com/blogs/opensource/machine-learning-with-autogluon-an-open-source-automl-library/)

**2. Documentation:** [Link](https://auto.gluon.ai/stable/index.html)

**3. GitHub Repository:** [Link](https://github.com/autogluon/autogluon)

In [19]:
# Prepare data function for AutoGluon: AutoGluon needs all the PCs and the target in a single DataFrame as its designed to work on Pandas dataframe.
def prepare_ag_data(X_pca, y_target):
    df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
    df['target'] = y_target.values
    return df

train_data_ag = prepare_ag_data(X_train_pca, y_train)
test_data_ag = prepare_ag_data(X_test_pca, y_test)

In [20]:
# Train AutoGluon and fit the model
predictor = TabularPredictor(
    label='target', # 'Log_BEV Percentage (Total Number Of Registrations)'
    problem_type='regression', # Prediction of continuous numbers
    eval_metric='mean_absolute_error' # 'mean_absolute_error' because R2 is unstable for single-year tests
).fit(
    train_data_ag, # Training data
    presets='best_quality', # Enables stacking and ensembling for higher accuracy
    time_limit=600          # 10-minute training limit. Will try to train as many model combinations as possible within this window.
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260111_082705"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct  2 10:42:05 UTC 2025
CPU Count:          2
Pytorch Version:    2.9.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       11.27 GB / 12.67 GB (89.0%)
Disk Space Avail:   85.65 GB / 107.72 GB (79.5%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stack

(_ray_fit pid=2466) [1000]	valid_set's l1: 0.321469
(_ray_fit pid=2466) [2000]	valid_set's l1: 0.314357
(_ray_fit pid=2466) [3000]	valid_set's l1: 0.313298
(_ray_fit pid=2466) [4000]	valid_set's l1: 0.312617


(_dystack pid=2054) 	-0.3101	 = Validation score   (-mean_absolute_error)
(_dystack pid=2054) 	27.69s	 = Training   runtime
(_dystack pid=2054) 	0.02s	 = Validation runtime
(_dystack pid=2054) Fitting model: LightGBM_BAG_L1 ... Training model for up to 52.99s of the 96.54s of remaining time.
(_dystack pid=2054) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.15%)
(_dystack pid=2054) 	-0.3137	 = Validation score   (-mean_absolute_error)
(_dystack pid=2054) 	22.32s	 = Training   runtime
(_dystack pid=2054) 	0.01s	 = Validation runtime
(_dystack pid=2054) Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 24.91s of the 68.47s of remaining time.
(_dystack pid=2054) 	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0
(_dystack pid=2054) 	-0.3056	 = Validation score   (-mean_absolute_error)
(_dystack pid=2054) 	2.05s	 = Training   runtime
(_dystack pid=2054) 	0.27s	 = V

(_ray_fit pid=3738) [1000]	valid_set's l1: 0.303292


(_dystack pid=2054) 	-0.3041	 = Validation score   (-mean_absolute_error)
(_dystack pid=2054) 	23.68s	 = Training   runtime
(_dystack pid=2054) 	0.03s	 = Validation runtime
(_dystack pid=2054) Fitting model: LightGBM_BAG_L2 ... Training model for up to 8.23s of the 8.14s of remaining time.
(_dystack pid=2054) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.16%)
(_ray_fit pid=4054) 	Ran out of time, early stopping on iteration 525. Best iteration is:
(_ray_fit pid=4054) 	[524]	valid_set's l1: 0.253507


(_ray_fit pid=4193) [1000]	valid_set's l1: 0.277606
(_ray_fit pid=4280) [1000]	valid_set's l1: 0.338902


(_dystack pid=2054) 	-0.3015	 = Validation score   (-mean_absolute_error)
(_dystack pid=2054) 	24.68s	 = Training   runtime
(_dystack pid=2054) 	0.04s	 = Validation runtime
(_dystack pid=2054) Fitting model: WeightedEnsemble_L3 ... Training model for up to 130.64s of the -23.01s of remaining time.
(_dystack pid=2054) 	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/9.6 GB
(_dystack pid=2054) 	Ensemble Weights: {'CatBoost_BAG_L1': 0.6, 'LightGBM_BAG_L2': 0.36, 'RandomForestMSE_BAG_L1': 0.04}
(_dystack pid=2054) 	-0.2838	 = Validation score   (-mean_absolute_error)
(_dystack pid=2054) 	0.17s	 = Training   runtime
(_dystack pid=2054) 	0.0s	 = Validation runtime
(_dystack pid=2054) AutoGluon training complete, total runtime = 153.86s ... Best model: WeightedEnsemble_L3 | Estimated inference throughput: 316.0 rows/s (36 batch size)
(_dystack pid=2054) TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/content/AutogluonModels/ag-20260111_082705/ds_su

In [21]:
# View the Model Leaderboard
# This shows you which models (XGBoost, Neural Nets, etc.) performed best within the time frame
lb = predictor.leaderboard(test_data_ag, silent=True)
print("\n--- Model Leaderboard ---")
print(lb)


--- Model Leaderboard ---
                        model  score_test  score_val          eval_metric  \
0        LightGBMLarge_BAG_L1   -0.676157  -0.337021  mean_absolute_error   
1        LightGBM_r131_BAG_L1   -0.687394  -0.309209  mean_absolute_error   
2        ExtraTreesMSE_BAG_L1   -0.695965  -0.294265  mean_absolute_error   
3             LightGBM_BAG_L1   -0.705985  -0.311912  mean_absolute_error   
4       NeuralNetTorch_BAG_L1   -0.720430  -0.275737  mean_absolute_error   
5      RandomForestMSE_BAG_L1   -0.730389  -0.303190  mean_absolute_error   
6      NeuralNetFastAI_BAG_L1   -0.730585  -0.333068  mean_absolute_error   
7              XGBoost_BAG_L1   -0.732744  -0.309043  mean_absolute_error   
8           LightGBMXT_BAG_L1   -0.842665  -0.301331  mean_absolute_error   
9         WeightedEnsemble_L2   -0.948871  -0.269547  mean_absolute_error   
10            CatBoost_BAG_L1   -0.964743  -0.277971  mean_absolute_error   
11       CatBoost_r177_BAG_L1   -1.037154  -0.285

**Explantion of the results**

The autogluon has checked on 13 different models

In [22]:
# Generate Predictions for 2020-2024
# We drop the 'target' column from the test data to predict it
# # Ensures the model is forced to guess the result using only the input features (PC1, PC2, PC3) without "cheating" by looking at the answer.
preds = predictor.predict(test_data_ag.drop(columns=['target']))

In [23]:
# Back-transform by converting log values back to real BEV percentages
y_test_real = np.exp(test_data_ag['target'])
preds_real = np.exp(preds)

In [24]:
# Create unified results dataframe
all_results = pd.DataFrame({
    'Country': post_covid_fixed['Country'],
    'Year': post_covid_fixed['Year'],
    'Actual_BEV_%': y_test_real,
    'Predicted_BEV_%': preds_real
})

# Absolute error
all_results['Error_Abs'] = (all_results['Actual_BEV_%'] - all_results['Predicted_BEV_%']).abs()

# Loop year-wise
for year, year_df in all_results.groupby('Year'):

    print(f"====>YEAR: {int(year)} PREDICTION RESULTS")

    # Leaders
    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    leaders_df = (
        year_df[year_df['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(leaders_df[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))

    # Followers
    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    followers_df = (
        year_df[year_df['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(followers_df[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))


====>YEAR: 2023 PREDICTION RESULTS

**** LEADERS CLUSTER (2023) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
        Sweden         39.42        15.396734  24.023266
       Denmark         37.24         8.285649  28.954351
       Finland         34.91         8.922422  25.987578
   Netherlands         31.52         9.535727  21.984273
       Austria         20.91        11.668066   9.241934
       Belgium         20.59         8.067822  12.522178
       Germany         19.28        14.402051   4.877949
        France         17.73        12.868250   4.861750
       Romania         11.53        13.969532   2.439532
         Spain          6.78         8.682150   1.902150
       Hungary          6.66        13.410844   6.750844
         Italy          5.23        12.264469   7.034469
        Poland          4.60        15.220713  10.620713
Czech Republic          4.39        15.427583  11.037583

**** FOLLOWERS CLUSTER (2023) ****
   Country  Actual_BEV_%  Predicted_BEV_

In [25]:
# Filter the results as per the clusters
all_results['Cluster'] = all_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_results_df = (
    all_results[all_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_results_df = (
    all_results[all_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_results_df.to_excel("Trial2_AutoGluon_Leaders_Results.xlsx", index=False)
followers_results_df.to_excel("Trial2_AutoGluon_Followers_Results.xlsx", index=False)